In [0]:
#objetivo dessa camada: unificar dois cadastros do mesmo assunto
#camada prata é onde fazemos parte de governança, auditoria, etc
#aqui garantimos que o dado tá com qualidade, tá como nós esperamos pra gente efetivamente conseguir fazer as análises.

#permitido na prata:

#tipagem | string vira `TIMESTAMP`, `INT`, `DATE` | 
#legibilidade | quebrar timestamp em data e hora | 
#metadados | `COMMENT` em toda coluna, tags na tabela | --> metadado é o dado sobre o dado --> dar contexto pra IA, só o dado solto ela pode alucinar. Dar contexto sobre o dado, ou seja, dado sobre o dado, é importante.
#unificação | dois cadastros do mesmo assunto, com a origem por registro | 
#aritmética pura | `atraso = real - previsto` | --> ao invés de fazer esse mesmo cálculo para cada análise na ouro, faço de uma vez na prata, sem alterar nenhuma coluna, só usando os dados que já temos e gerando uma nova. 

#não permitido na prata: 

#filtro / `WHERE` de negócio |
#`GROUP BY` / agregação | 
#limiar, flag, classificação |

#sem eliminação de dados. Transformando dados da bronze em dados com governança. Estamos garantindo a qualidade do dado. Eliminação de dados por conta de regras de negócio só na camada ouro. Objetivo aqui é deixar dados prontos para que eu possa consumi-los na ouro pra fazer análises de negócios.
#Na bronze, objetivos é deixar dados presentes, pegar os dados como sao, colocar no ambiente. Na silver, dar qualidade e capacidade pra eles serem consumidos para aspectos de negócios.

#depois da ingestão, temos que fazer uma análise exploratória. Ver problemas na origem e na tabela bronze sobre nulidade, sobre strings estranhas, etc.

In [0]:
#começando, vamos olhar para algumas colunas que podem ter dados nulos. Iniciando a analise exploratória
display(spark.sql("""
    SELECT
      COUNT(*)                                                        AS linhas,
      SUM(CASE WHEN partida_real     IS NULL THEN 1 ELSE 0 END)       AS partida_real_null_de_verdade,
      SUM(CASE WHEN partida_real     = 'null' THEN 1 ELSE 0 END)      AS partida_real_string_null,
      SUM(CASE WHEN partida_prevista = 'null' THEN 1 ELSE 0 END)      AS partida_prevista_string_null,
      SUM(CASE WHEN partida_prevista LIKE '%.%' THEN 1 ELSE 0 END)    AS com_fracao_de_segundo
    FROM voebem.bronze.vra
"""))
#aqui em cima é que, se tiver dado nulo, ele retornará 0
#Isso não é regra de negócio. Não podemos ter dados nulos, isso é princípio básico, questão de qualidade dos dados.
# no fim, vai mostrar quantas linhas estão nulas em algumas colunas da nossa tabela vra

In [0]:
%sql

--precisamos criar um schema da camada prata
CREATE SCHEMA IF NOT EXISTS voebem.silver

In [0]:
#query em algumas colunas que queremos. O try_cast ele vai tentar transformar o que for nulo das colunas em timestamp, porque vão ser usados pra cálculo então não queremos o dado nulo.


#Armadilha 1 — A string 'null' (4 caracteres): WHERE partida_real IS NULL devolve zero numa tabela onde 29 mil voos não têm horário real. Correção: nullif(coluna, 'null') antes do cast. 

# Armadilha 2 — Múltiplos formatos de timestamp no mesmo arquivo: A maioria das linhas segue o padrão 2026-01-27 19:45:00, mas ~80 mil linhas vêm com fração de segundo de 9 casas. Um to_timestamp(col, 'yyyy-MM-dd HH:mm:ss') fixo devolveria NULL silenciosamente para 8% da base. O try_cast(... AS TIMESTAMP) aceita ambos os formatos, e o prefixo try_ garante que novos formatos virem NULL em vez de derrubar o job.

#Nota: Trata-se de tipagem, não limpeza de negócio. Traduzir 'null' para NULL apenas expressa a ausência de dado no tipo correto, sem descartar nenhuma linha.

#Repare no que não existe nesta query: nenhum WHERE, nenhum GROUP BY, nenhum DISTINCT, nenhum JOIN. É um SELECT de projeção sobre o bronze inteiro. E repare nas três colunas do fim: atraso_partida_min, atraso_chegada_min e minutos_recuperados. São subtrações entre colunas da própria linha. Não têm limiar, não classificam nada, não escondem número mágico — e, principalmente, não impedem análise nenhuma. Por isso podem morar aqui.

spark.sql("""
CREATE OR REPLACE TABLE voebem.silver.vra AS
WITH tipado AS (
  SELECT
    icao_empresa,
    numero_voo,
    codigo_di,
    codigo_tipo_linha,
    icao_origem,
    icao_destino,
    try_cast(nullif(partida_prevista, 'null') AS TIMESTAMP) AS partida_prevista,
    try_cast(nullif(partida_real,     'null') AS TIMESTAMP) AS partida_real,
    try_cast(nullif(chegada_prevista, 'null') AS TIMESTAMP) AS chegada_prevista,
    try_cast(nullif(chegada_real,     'null') AS TIMESTAMP) AS chegada_real,
    situacao_voo,
    nullif(codigo_justificativa, 'N/A')                     AS codigo_justificativa, --quando código_justificativa tiver nulo, vai ser colocado o 'N/A'
    _arquivo_origem,
    _ingerido_em
  FROM voebem.bronze.vra
)

--Segundo select usando nossa cte que construímos acima. cte foi usada pq é mais performática e mais legível. não preciso criar outra tabela ou dataframe pra seguir, cte supre isso.
SELECT
  icao_empresa,
  numero_voo,
  codigo_di,
  codigo_tipo_linha,
  icao_origem,
  icao_destino,

  partida_prevista,
  CAST(partida_prevista AS DATE)                     AS partida_prevista_data, 
  date_format(partida_prevista, 'HH:mm')             AS partida_prevista_hora, --cast quer dizer transformar dado, transformando de string para date, e o date_format mostra que é hora e minuto. tamos mudando a tipagem do dado

  partida_real,
  CAST(partida_real AS DATE)                         AS partida_real_data,
  date_format(partida_real, 'HH:mm')                 AS partida_real_hora,

  chegada_prevista,
  CAST(chegada_prevista AS DATE)                     AS chegada_prevista_data,
  date_format(chegada_prevista, 'HH:mm')             AS chegada_prevista_hora,

  chegada_real,
  CAST(chegada_real AS DATE)                         AS chegada_real_data,
  date_format(chegada_real, 'HH:mm')                 AS chegada_real_hora,

  situacao_voo,
  codigo_justificativa,

  -- aritmetica pura: subtracao de colunas da propria linha, sem limiar e sem decisao
  CAST(timestampdiff(MINUTE, partida_prevista, partida_real) AS INT) AS atraso_partida_min,
  CAST(timestampdiff(MINUTE, chegada_prevista, chegada_real) AS INT) AS atraso_chegada_min,
  CAST(timestampdiff(MINUTE, partida_prevista, partida_real)
     - timestampdiff(MINUTE, chegada_prevista, chegada_real) AS INT) AS minutos_recuperados,

  _arquivo_origem,
  _ingerido_em,
  current_timestamp()                                AS _transformado_em
FROM tipado
""")

print("silver.vra criada")

#pyspark usa sql ansi, o tipo mais primitivo e simples existente. acima, usamos CTEs que é parte do padrão ansi. CTE é uma gestao de tabela temporária. a saída do select dessa tabela vai gerar dados tipados, e esses dados vao virar uma tabela temporária
#timestampdiff e MINUTE são funções que foram pré-disponibilizadas dentro do sql pelo pyspark
#cuidado com scripts que precisam de acesso externo, ou seja, que nao rodam puramente offline! é uma brecha de segurança
#então finalizada a parte inicial da silver, onde fizemos algumas tipagens específicas, criamos algumas colunas novas pra guardar as horas e fizemos cálculos pros atrasos